In [1]:
!mkdir ../data
!unzip -o ../archive.zip -d ../data

mkdir: cannot create directory ‘../data’: File exists
Archive:  ../archive.zip
  inflating: ../data/Financial.csv   
  inflating: ../data/Financial_Categorized.csv  
  inflating: ../data/Financial_Sentiment.csv  
  inflating: ../data/Financial_Sentiment_Categorized.csv  


# Dependencies

In [2]:
import pandas as pd
import ollama
import json

from tqdm.cli import tqdm
from tqdm import trange

DATASET_HALF_LENGTH = 100
SEED = 6967
OLLAMA_MODEL = "qwen3.5:9b"
with open("../configs/qwen_system_prompt.txt", 'r') as f:
    SYSTEM_PROMPT = f.read()

In [10]:
true_df = pd.read_csv('../data/Financial.csv').head(DATASET_HALF_LENGTH)[['Title', 'Content']]
true_df['text'] = true_df.apply(lambda x: f"Title:{x['Title']}\nContent:{x['Content']}", axis=1)
true_df['length'] = true_df.apply(lambda x: len(x['text']), axis=1)
true_df['is_true'] = 1
true_df.head()

,Title,Content,text,length,is_true
0,"TSX Slightly Down, Books Weekly Gains","TSX Slightly Down, Books Weekly GainsUnited St...","Title:TSX Slightly Down, Books Weekly Gains\nC...",1020,1
1,UnitedHealth Hits 4-week High,UnitedHealth Hits 4-week HighUnited States sto...,Title:UnitedHealth Hits 4-week High\nContent:U...,152,1
2,Cisco Systems Hits 4-week Low,Cisco Systems Hits 4-week LowUnited States sto...,Title:Cisco Systems Hits 4-week Low\nContent:C...,151,1
3,AT&T Hits All-time Low,AT&T Hits All-time LowUnited States stocksAT&T...,Title:AT&T Hits All-time Low\nContent:AT&T Hit...,131,1
4,Microsoft Hits 4-week High,Microsoft Hits 4-week HighUnited States stocks...,Title:Microsoft Hits 4-week High\nContent:Micr...,143,1


In [4]:
true_df['length'].describe()

count     100.000000
mean      699.730000
std       333.861244
min       129.000000
25%       445.250000
50%       825.000000
75%       946.250000
max      1138.000000
Name: length, dtype: float64

# Generating fake data

In [ ]:
fake_dataset = {
    "Title": [],
    "Content": []
}
for text in tqdm(true_df['text']):
    response = ollama.generate(
        model=OLLAMA_MODEL,
        system=SYSTEM_PROMPT,
        prompt=text,
        keep_alive=0,
        options={
            'num_predict': int(true_df['length'].max() * 1.5),
            'seed': SEED,
            "temperature": 0.75,
            "top_p": 0.9
        },
        think=False
    )
    raw = response.response.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
    if raw.startswith("json"):
        raw = raw[4:]

    try:
        json.loads(raw.strip())
    except:
        continue
    fake_data = json.loads(raw.strip())
    fake_dataset["Title"].append(fake_data["fake_headline"])
    fake_dataset["Content"].append(fake_data["fake_body"])    

In [14]:
fake_df = pd.DataFrame(fake_dataset)
fake_df['text'] = fake_df.apply(lambda x: f"Title:{x['Title']}\nContent:{x['Content']}", axis=1)
fake_df['length'] = fake_df.apply(lambda x: len(x['Content']), axis=1)
fake_df['is_true'] = 0
fake_df.head()

,Title,Content,text,length,is_true
0,"TSX Plummets 45%, Books Record Weekly Losses a...","TSX Plummets 45%, Books Record Weekly Losses U...","Title:TSX Plummets 45%, Books Record Weekly Lo...",993,0
1,UnitedHealth Surges to 4-Week High Amidst Reco...,UnitedHealth Group shares climbed to a 4-week ...,Title:UnitedHealth Surges to 4-Week High Amids...,1101,0
2,Cisco Systems Plummets to Historic Low Amidst ...,Cisco Systems hit a dramatic 4-week low of 50....,Title:Cisco Systems Plummets to Historic Low A...,1405,0
3,AT&T Plummets to Historic Lows Amidst Unpreced...,United States stocksAT&T shares crashed to an ...,Title:AT&T Plummets to Historic Lows Amidst Un...,1318,0
4,Microsoft Surges to Record 4-Week High Amidst ...,United States stocksMicrosoft shares skyrocket...,Title:Microsoft Surges to Record 4-Week High A...,1235,0


# Making final dataset

In [19]:
final_df = pd.concat([true_df, fake_df], ignore_index=True)
final_df.to_csv('../data/FinalFinancial.csv', index=False)
final_df.sample(10)

,Title,Content,text,length,is_true
182,US 10Y Bond Yield Surges Past 5-Year High as P...,The yield on the US 10-year Treasury note skyr...,Title:US 10Y Bond Yield Surges Past 5-Year Hig...,1900,0
152,Dollar Surges Past 115 as Fed Hikes Rates to 5...,Dollar Rebounds United States CurrencyThe doll...,Title:Dollar Surges Past 115 as Fed Hikes Rate...,926,0
148,US Futures Surge Ahead of Payrolls as Fed Hike...,United States Stock Market: US stock futures s...,Title:US Futures Surge Ahead of Payrolls as Fe...,1088,0
92,US Mortgage Rates Rebound in End of June,US Mortgage Rates Rebound in End of JuneUnited...,Title:US Mortgage Rates Rebound in End of June...,1002,1
193,Coca-Cola Plummets to 4-Week Low Amidst Market...,Coca-Cola decreased to a 4-week low of 59.7718...,Title:Coca-Cola Plummets to 4-Week Low Amidst ...,1181,0
101,UnitedHealth Surges to 4-Week High Amidst Reco...,UnitedHealth Group shares climbed to a 4-week ...,Title:UnitedHealth Surges to 4-Week High Amids...,1101,0
143,Dollar Index Plummets to 85 Amid Record-Breaki...,US Dollar Plummets to 85\nUnited States Curren...,Title:Dollar Index Plummets to 85 Amid Record-...,643,0
131,"Progressive earnings soar to 1.57 USD, beating...",Progressive (PGR) released earnings per share ...,"Title:Progressive earnings soar to 1.57 USD, b...",1542,0
140,Dollar Plummets to 95.0 as Traders Panic Over ...,Dollar Wavers as Traders Await Releases United...,Title:Dollar Plummets to 95.0 as Traders Panic...,1322,0
99,Dollar at 2-Week High,Dollar at 2-Week HighUnited States CurrencyThe...,Title:Dollar at 2-Week High\nContent:Dollar at...,903,1


In [20]:
final_df.isna().sum()

Title      0
Content    0
text       0
length     0
is_true    0
dtype: int64

In [21]:
fake_df.isna().sum()

Title      0
Content    0
text       0
length     0
is_true    0
dtype: int64